# Literature-to-Figure Workflow

このnotebookは、文献整理から `Figure 1` / `Figure 2` に至る処理の流れを一枚の作業記録としてまとめる。

対象の流れ:

1. 文献まとめ
2. 文献から `width`, `depth`, `encoding`, `backend` などを抽出
3. 抽出結果を `200_20260625_circuit_resources.csv` に集約
4. CSVから `Figure 1` を作成
5. QRL-like stage に分類
6. `width` と `depth` の対応を作成
7. `Figure 2` を作成

主要な入力・出力:

- Input: `literature/200_20260625_circuit_resources.csv`
- Evidence notes: `literature/12_circuit_resource_sources.md`
- Figure 1 script: `05_src/visualization/make_presentation_figure_1.py`
- Figure 2 script: `scripts/make_presentation_figure_2_arl23.py`
- Figure 1 data: `06_outputs/figures/active/figure_1_presentation_plot_data.csv`
- Figure 2 data: `06_outputs/figures/active/figure_2_arl2_arl3_plot_data.csv`

In [ ]:
from pathlib import Path
import csv
import re
from collections import Counter, defaultdict

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

LITERATURE = ROOT / "literature"
FIGURES = ROOT / "figures"
SCRIPTS = ROOT / "scripts"

CIRCUIT_RESOURCES = LITERATURE / "200_20260625_circuit_resources.csv"
SOURCE_NOTES = LITERATURE / "12_circuit_resource_sources.md"
FIGURE1_CSV = FIGURES / "figure_1_presentation_plot_data.csv"
FIGURE2_CSV = FIGURES / "figure_2_arl2_arl3_plot_data.csv"
FIGURE1_SCRIPT = SCRIPTS / "make_presentation_figure_1.py"
FIGURE2_SCRIPT = SCRIPTS / "make_presentation_figure_2_arl23.py"

paths = [
    CIRCUIT_RESOURCES,
    SOURCE_NOTES,
    FIGURE1_CSV,
    FIGURE2_CSV,
    FIGURE1_SCRIPT,
    FIGURE2_SCRIPT,
]

for path in paths:
    print(f"{path.relative_to(ROOT)}: {'ok' if path.exists() else 'missing'}")

## 1. 文献まとめ

まず対象文献を役割別に整理する。ここでは各文献を、理論定義、benchmark方法論、simulation、hardware-aware evidence、resource estimation などの役割に分ける。

この段階の目的は、すべての文献を同じ種類の証拠として扱わないこと。たとえば、QAOA原典の `n logical qubits` は理論定義であり、VRP/CVRP個別インスタンスの実験値ではない。

In [ ]:
def read_csv_dicts(path):
    with path.open(newline="", encoding="utf-8-sig") as f:
        return list(csv.DictReader(f))

resources = read_csv_dicts(CIRCUIT_RESOURCES)

print(f"rows: {len(resources)}")
print("extraction_status counts:")
for status, count in Counter(row["extraction_status"] for row in resources).most_common():
    print(f"  {status}: {count}")

print("\npapers in 200_20260625_circuit_resources.csv:")
for paper_id in sorted({row["paper_id"] for row in resources}):
    print(f"  {paper_id}")

## Researcher-defined rules and settings

このnotebookでは、文献から直接引用した値と、本研究で分析のために設定した操作的定義を分けて扱う。

### 文献由来の値

以下は原則として論文本文、表、図、または論文中の式から抽出した値である。

- `problem`: 論文が扱う問題種別。例: VRP, CVRP, HVRP, VRPTW。
- `instance_or_scope`: 論文で報告された対象インスタンス、実験設定、または理論的範囲。
- `formulation_or_encoding`: 論文中の定式化・encoding。例: QUBO, HOBO, Ising mapping, full encoding, minimal encoding。
- `binary_variables_or_problem_size`: 顧客数、車両数、route数、binary variable数など、論文が示す問題規模。
- `circuit_width_qubits`: 論文が報告したqubit数、または論文中の明示式から整理したqubit数。
- `circuit_depth`: 論文が報告したdepth、QAOA layer `p`、ansatz layer `L`、または理論depth。
- `hardware_or_backend`: 論文が報告したsimulator、IBM、Rigetti、IonQ、resource estimation等。
- `source_location`: 抽出根拠となる節、図、表、または確認箇所。

### 本研究で設定した定義・ルール

以下は、比較・可視化のために本研究側で設定したルールであり、個別論文がそのまま使っている分類とは限らない。

- `width_numeric`: `circuit_width_qubits` が `11 qubits` のように具体的な数値で始まる場合だけ、数値部分を抽出したもの。`N^2 qubits` や `n logical qubits` はFigure 1の数値棒には入れない。
- `depth_upper`: `p = 1 to 5` や `p = 6 to 40` のような範囲表記から最大値を取ったもの。これは物理的なtranspiled depthではなく、報告されたdepth/layer parameterの上限値である。
- `validation_stage`: Figure 1用に、backendや抽出状態から `Resource estimation`, `Hardware run / target`, `Simulator + hardware check`, `Classical simulation` に整理した分類。
- `QRL-like stage`: Figure 2用に、証拠の段階を `QRL2-like`, `QRL3-like`, `QRL4-like`, `definition/method` として再分類したもの。これは正式な標準QRLではなく、本研究内の比較用ラベルである。
- Figure 1 inclusion rule: 具体的な数値qubit数を抽出できる行だけを表示する。理論式・方法論だけの行は除外する。
- Figure 2 inclusion rule: `width_numeric` と `depth_upper` の両方があり、かつ `QRL2-like` または `QRL3-like` の行に限定する。
- Sorting rule: Figure 1では `Resource estimation` -> `Hardware run / target` -> `Simulator + hardware check` -> `Classical simulation` の順に並べ、各stage内ではwidthが大きい順に並べる。
- Same-instance handling: 同じ `instance_or_scope` でも `formulation_or_encoding` が違えば別行として扱う。例: Golden_5 QUBOとGolden_5 HOBO、16-route full encodingと16-route minimal encoding。

### 解釈上の制約

- `width` は必要qubit数であり、実用性や優位性を単独で示す指標ではない。
- `depth_upper` は文献ごとのdepth/layer表記を可視化用にそろえた値であり、異なるdepth定義を完全に正規化したものではない。
- hardware run, simulator, resource estimationは証拠の強さ・意味が異なるため、同じ図に載せる場合もstage分類を併記する。
- このworkflowは既存研究の技術証拠を整理するためのものであり、量子計算がVRP/CVRPで実用的優位を持つことを示すものではない。

## 2. 文献から抽出

各論文から以下を抽出し、`200_20260625_circuit_resources.csv` に記録する。

- `paper_id`: 論文ID
- `problem`: VRP, CVRP, HVRP, VRPTW など
- `instance_or_scope`: 対象インスタンスまたは分析範囲
- `formulation_or_encoding`: QUBO, HOBO, Ising, full encoding, minimal encoding など
- `binary_variables_or_problem_size`: 顧客数、車両数、route数、binary variable数など
- `circuit_width_qubits`: 回路幅。基本的には量子回路が作用するqubit数
- `circuit_depth`: 論文が報告する深さ、層数、またはdepth parameter
- `depth_definition`: `p`, `L`, theoretical depth, transpiled depth などの意味
- `hardware_or_backend`: simulator, IBM, Rigetti, IonQ, resource estimation など
- `source_location`: 図表番号、節名、または確認箇所
- `notes`: 解釈上の注意

抽出根拠は `literature/12_circuit_resource_sources.md` に残す。

In [ ]:
preview_columns = [
    "paper_id",
    "problem",
    "instance_or_scope",
    "formulation_or_encoding",
    "circuit_width_qubits",
    "circuit_depth",
    "depth_definition",
    "hardware_or_backend",
    "source_location",
]

for row in resources[:8]:
    print("---")
    for column in preview_columns:
        print(f"{column}: {row.get(column, '')}")

## 3. 抽出したものをCSVにまとめる

`200_20260625_circuit_resources.csv` は文献から抽出した一次整理表である。

重要な扱い:

- 同じ論文でも、インスタンスやencodingが違えば別行にする。
- 同じインスタンス名でも、encodingが違えば `width` は変わりうる。
- `depth` は文献ごとに意味が違うため、必ず `depth_definition` と一緒に扱う。
- 理論式だけの `N^2 qubits` や `n logical qubits` は、数値比較用のFigure 1には直接入れない。

In [ ]:
def parse_width(value):
    if value is None:
        return None
    match = re.search(r"^\s*([0-9][0-9,]*)\s*qubits?", str(value))
    if match:
        return int(match.group(1).replace(",", ""))
    return None

def parse_depth_upper(value):
    if value is None:
        return None
    text = str(value).strip()
    if not text:
        return None
    numbers = [int(x.replace(",", "")) for x in re.findall(r"\d[\d,]*", text)]
    if not numbers:
        return None
    return max(numbers)

for row in resources:
    row["width_numeric"] = parse_width(row.get("circuit_width_qubits"))
    row["depth_upper"] = parse_depth_upper(row.get("circuit_depth"))

numeric_width_rows = [row for row in resources if row["width_numeric"] is not None]
numeric_width_depth_rows = [row for row in resources if row["width_numeric"] is not None and row["depth_upper"] is not None]

print(f"numeric width rows: {len(numeric_width_rows)}")
print(f"numeric width+depth rows: {len(numeric_width_depth_rows)}")

print("\nRows excluded from Figure 1 numeric bars because width is not a concrete number:")
for row in resources:
    if row["width_numeric"] is None:
        print(f"  {row['paper_id']} | {row['instance_or_scope']} | {row['circuit_width_qubits']}")

## 4. CSVからFigure 1化

`Figure 1` は、具体的な数値qubit数を抽出できる行だけを使う。

処理:

1. `circuit_width_qubits` から数値を抽出して `width_numeric` を作る。
2. `hardware_or_backend`, `extraction_status`, `notes`, `source_location` から `validation_stage` を付ける。
3. `Resource estimation`, `Hardware run / target`, `Simulator + hardware check`, `Classical simulation` の順に並べる。
4. 同じインスタンスでもencoding差がわかるように、Figure 1ラベルには `Encoding: ...` を併記する。

実際のFigure 1は `05_src/visualization/make_presentation_figure_1.py` で生成する。


研究者側で設定した可視化ルール: Figure 1では具体的な数値qubit数が抽出できる行のみを使い、validation stageはbackend/evidence情報から本研究内で再分類する。

In [ ]:
def validation_stage(row):
    backend = str(row.get("hardware_or_backend", "")).lower()
    status = str(row.get("extraction_status", "")).lower()
    notes = str(row.get("notes", "")).lower()
    source = str(row.get("source_location", "")).lower()

    is_simulation = any(token in backend for token in ["simulation", "simulator", "statevector", "qasm", "qiskit simulation"])
    has_hardware_name = any(token in backend for token in ["ibm", "rigetti", "ionq", "quantum system", "hardware"])

    if "resource estimation" in backend:
        return "Resource estimation"
    if has_hardware_name and is_simulation:
        return "Simulator + hardware check"
    if has_hardware_name:
        return "Hardware run / target"
    if is_simulation:
        return "Classical simulation"
    if "theory" in backend or status in {"definition", "methodology"} or "table 1" in source or "theoretical" in notes:
        return "Theory / method"
    return "Other"

stage_order = ["Resource estimation", "Hardware run / target", "Simulator + hardware check", "Classical simulation", "Theory / method", "Other"]
stage_rank = {stage: index for index, stage in enumerate(stage_order)}

figure1_rows = []
for row in numeric_width_rows:
    item = dict(row)
    item["validation_stage"] = validation_stage(row)
    figure1_rows.append(item)

figure1_rows = sorted(figure1_rows, key=lambda row: (stage_rank.get(row["validation_stage"], 99), -row["width_numeric"], row["paper_id"]))

print(f"Figure 1 rows: {len(figure1_rows)}")
for row in figure1_rows[:8]:
    print(f"{row['validation_stage']:28s} | {row['width_numeric']:>7} | {row['instance_or_scope']} | {row['formulation_or_encoding']}")

Figure 1生成コマンド:

```bash
.venv/bin/python 05_src/visualization/make_presentation_figure_1.py
```

出力:

- `06_outputs/figures/active/figure_1_width_by_validation_stage_presentation.png`
- `06_outputs/figures/active/figure_1_width_by_instance.png`
- `06_outputs/figures/active/figure_1_presentation_plot_data.csv`

In [ ]:
if FIGURE1_CSV.exists():
    figure1_csv_rows = read_csv_dicts(FIGURE1_CSV)
    print(f"Figure 1 CSV rows: {len(figure1_csv_rows)}")
    print("columns:", list(figure1_csv_rows[0].keys()) if figure1_csv_rows else [])
else:
    print("Figure 1 CSV has not been generated yet.")

## 5. QRL別化

`Figure 2` では `width` と `depth` の対応を見るため、主に次のQRL-like分類を使う。

- `QRL2-like: simulation evidence`: 古典シミュレータ上の量子アルゴリズム評価
- `QRL3-like: hardware-aware evidence`: 実機・特定backend・hardware-aware transpilationを含む証拠
- `QRL4-like: resource estimate`: 実行ではなく資源推定
- `definition/method`: 理論定義または方法論

Figure 2では比較可能性のため、QRL2/QRL3-likeのうち `width` と `depth` の両方が数値化できる行に絞る。


注意: ここでの `QRL-like` は本研究内の分析ラベルであり、外部標準として確定した公式分類ではない。

In [ ]:
def qrl_like_stage(row):
    backend = str(row.get("hardware_or_backend", "")).lower()
    status = str(row.get("extraction_status", "")).lower()

    if "resource estimation" in backend:
        return "QRL4-like: resource estimate"
    if status in {"definition", "methodology"} or "theory" in backend:
        return "definition/method"

    is_simulation = any(token in backend for token in ["simulation", "simulator", "statevector", "qasm", "qiskit simulation"])
    has_hardware = any(token in backend for token in ["ibm", "rigetti", "ionq", "quantum system", "hardware", "backend"])

    if has_hardware:
        return "QRL3-like: hardware-aware evidence"
    if is_simulation:
        return "QRL2-like: simulation evidence"
    return "other"

for row in resources:
    row["qrl_like_stage"] = qrl_like_stage(row)

print("QRL-like stage counts:")
for stage, count in Counter(row["qrl_like_stage"] for row in resources).most_common():
    print(f"  {stage}: {count}")

## 6. widthとdepthの対応付け

`width` は `circuit_width_qubits` から数値抽出する。

`depth` は文献ごとに意味が異なるため、Figure 2では単一の物理depthとしてではなく、報告された `p`, `L`, depth値の上限値を `depth_upper` として扱う。

例:

- `p = 1 to 5` → `depth_upper = 5`
- `p = 6 to 40` → `depth_upper = 40`
- `L = 4` → `depth_upper = 4`
- `p = 2` → `depth_upper = 2`

注意: `depth_upper` はbackend-transpiled physical depthではない。比較時は `depth_definition` を必ず確認する。


研究者側で設定した対応付け: Figure 2では `p`, `L`, 範囲表記などから抽出可能な最大値を `depth_upper` として置き、横軸の `width_numeric` と対応させる。

In [ ]:
keep_qrl_stages = {"QRL2-like: simulation evidence", "QRL3-like: hardware-aware evidence"}
figure2_rows = [
    row for row in resources
    if row["width_numeric"] is not None
    and row["depth_upper"] is not None
    and row["qrl_like_stage"] in keep_qrl_stages
]

figure2_rows = sorted(figure2_rows, key=lambda row: (row["qrl_like_stage"], row["width_numeric"], row["depth_upper"], row["paper_id"]))

print(f"Figure 2 rows: {len(figure2_rows)}")
for row in figure2_rows:
    print(f"{row['qrl_like_stage']:38s} | width={row['width_numeric']:>3} | depth_upper={row['depth_upper']:>2} | {row['paper_id']} | {row['instance_or_scope']}")

## 7. Figure 2化

`Figure 2` は、QRL2/QRL3-likeの行に限定して `width_numeric` と `depth_upper` の対応を見る図である。

Figure 2生成コマンド:

```bash
.venv/bin/python scripts/make_presentation_figure_2_arl23.py
```

ただし、現在の `make_presentation_figure_2_arl23.py` は `pandas` に依存している。環境に `pandas` がない場合は、`.venv` に追加するか、Figure 1 script と同様に標準 `csv` ベースへ移植する。

出力:

- `06_outputs/figures/active/figure_2_width_depth_arl2_arl3_presentation.png`
- `06_outputs/figures/active/figure_2_arl2_arl3_plot_data.csv`

In [ ]:
if FIGURE2_CSV.exists():
    figure2_csv_rows = read_csv_dicts(FIGURE2_CSV)
    print(f"Figure 2 CSV rows: {len(figure2_csv_rows)}")
    print("columns:", list(figure2_csv_rows[0].keys()) if figure2_csv_rows else [])
else:
    print("Figure 2 CSV has not been generated yet.")

## Summary

このnotebookでまとめた処理は次の通り。

```text
文献まとめ
  -> 文献からwidth/depth/encoding/backend等を抽出
  -> 200_20260625_circuit_resources.csvに集約
  -> concrete numeric widthだけを抽出してFigure 1化
  -> backend/evidence typeからQRL-like stageへ分類
  -> width_numericとdepth_upperを対応付け
  -> QRL2/QRL3-like行に限定してFigure 2化
```

Figure 1は「どの技術検証段階で、どの規模のqubit幅が報告されているか」を見る図。

Figure 2は「simulation / hardware-aware段階で、widthと報告depth parameterがどのように対応しているか」を見る図。

どちらも、量子計算が実用的に優位であることを示す図ではなく、既存研究の技術証拠の段階と規模を整理するための中間証拠である。